In [0]:
import sys
from datetime import datetime, timedelta
from pyspark.sql.functions import min as spark_min, max as spark_max
from concurrent.futures import ThreadPoolExecutor, as_completed

# 1. Widget Setup
dbutils.widgets.text("customer_id", "cust_001")
dbutils.widgets.text("object_name", "events")
dbutils.widgets.text("source_system", "bigquery")
dbutils.widgets.text("bucket_path", "s3://hgi-databricks-data-lakehouse-dev/")
dbutils.widgets.text("bq_catalog_table", "`bigquery-connection_catalog`.`v4c_bigquery_dataset`.`event_raw`")

customer_id = dbutils.widgets.get("customer_id")
object_name = dbutils.widgets.get("object_name")
source_system = dbutils.widgets.get("source_system")
bucket_path = dbutils.widgets.get("bucket_path")
bq_catalog_table = dbutils.widgets.get("bq_catalog_table")

now = datetime.now()
yyyy, mm, dd, hh = now.strftime("%Y"), now.strftime("%m"), now.strftime("%d"), now.strftime("%H")
historical_s3_path = f"{bucket_path}landing-zone/{source_system}/{customer_id}/{object_name}/historical/{yyyy}/{mm}/{dd}/{hh}"

print(f"🚀 Starting Historical Load for {customer_id} - {object_name}")

try:
    # 2. Find Min/Max (Only hits BigQuery once)
    bounds_df = spark.table(bq_catalog_table).select(
        spark_min("event_timestamp").alias("min_ts"),
        spark_max("event_timestamp").alias("max_ts")
    )
    bounds = bounds_df.collect()[0]
    
    start_date = bounds["min_ts"]
    end_date = bounds["max_ts"]
    
    if start_date is None or end_date is None:
        print("No data found in source.")
        dbutils.notebook.exit("0") 
        
    print(f"Data range: {start_date} to {end_date}")

    # 3. Create 1-Hour Intervals
    intervals = []
    current_start = start_date
    chunk_interval = timedelta(hours=1)
    
    while current_start <= (end_date + chunk_interval):
        current_end = current_start + chunk_interval
        intervals.append((current_start.strftime("%Y-%m-%d %H:%M:%S"), current_end.strftime("%Y-%m-%d %H:%M:%S")))
        current_start = current_end

    print(f"Divided timeline into {len(intervals)} hourly chunks.")

    # 4. Processing Function (Optimized for no double-scans)
    def process_chunk(interval):
        start_str, end_str = interval
        try:
            df_chunk = spark.table(bq_catalog_table) \
                .filter(f"event_timestamp >= TIMESTAMP('{start_str}') AND event_timestamp < TIMESTAMP('{end_str}')")
            
            df_chunk.write.mode("append").format("parquet").option("compression","snappy").save(historical_s3_path)
            return f"Processed {start_str} to {end_str}"
            
        except Exception as e:
            print(f"Failed chunk {start_str}: {str(e)}")
            raise e

    # 5. Execute in Parallel (Max 4 for 2-core Serverless)
    max_workers = 4
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_chunk, interval): interval for interval in intervals}
        for future in as_completed(futures):
            print(f"✅ {future.result()}")

    print("✅ Historical Data Load FINISHED!")

    # 6. Initialize Watermark with 1-Minute Lookback
    print(f"Initializing watermark to {end_date} minus 1 minute...")
    
    spark.sql("""
        CREATE TABLE IF NOT EXISTS ingestion_metadata.watermarks (
            source_system STRING,
            object_name STRING,
            last_processed_timestamp TIMESTAMP
        ) USING DELTA
    """)
    
    # Notice the `- INTERVAL 1 MINUTE` added here
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW new_watermarks AS
        SELECT source_system, object_name, last_processed_timestamp
        FROM ingestion_metadata.watermarks
        WHERE NOT (source_system = '{source_system}' AND object_name = '{object_name}')
        UNION ALL
        SELECT '{source_system}' AS source_system, '{object_name}' AS object_name, TIMESTAMP('{end_date}') - INTERVAL 1 MINUTE AS last_processed_timestamp
    """)
    
    spark.sql("INSERT OVERWRITE TABLE ingestion_metadata.watermarks SELECT * FROM new_watermarks")
    print("✅ Watermark initialized successfully with lookback window.")
    
    dbutils.notebook.exit("Success")

except Exception as e:
    print(f"❌ Pipeline Failed: {str(e)}")
    raise e